# M5 · Clickstream Cleaning (pandas, chunked)

**Goal:** clean all 500,000 web events. The file (122 MB) is processed in **chunks of 100,000 rows**: every chunk goes through the same cleaning function, so memory stays low and still **every row is cleaned** (no sampling).

Every step refers to an `issue_id` from the M1 issue log. Values that cannot be proven are set to null and flagged.

**Outputs**
| File | Folder | Content |
|---|---|---|
| `clickstream_clean.xlsx` | `clean_data/` | sheets `clickstream` (all 500,000 clean events; fits Excel's 1,048,576-row limit) and `flagged` (one row per event problem, with the raw value) |
| `clickstream_clean.zip` | `clean_data/csv/` | the same clean events as UTF-8 CSV, zipped (the plain CSV is larger than GitHub's 100 MB limit) |
| `m5_fix_log.xlsx` | `output/` (this folder) | fixes linked to M1 issues, validation |

## 1. Setup

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

RAW_DIR = Path("../../01_data_and_task/raw_data")
CLEAN_DIR = Path("../../03_clean_data_and_report/clean_data")
OUT_DIR = Path("output")  # detailed working outputs of this milestone (logs, charts)
OUT_DIR.mkdir(exist_ok=True)
ISSUE_LOG = Path("../01_profiling-pandas/output/issue_log.xlsx")
RAW_FILE = RAW_DIR / "clickstream_500k_events.csv"
CHUNK_SIZE = 100_000
READ_OPTS = dict(dtype=str, keep_default_na=False, encoding="utf-8")

# The raw files were exported on 2025-12-02; an event after that day cannot be real.
AFTER_SNAPSHOT = pd.Timestamp("2025-12-03", tz="UTC")

customers = pd.read_excel(CLEAN_DIR / "customers_clean.xlsx", sheet_name="customers", usecols=["customer_id"])
products = pd.read_excel(CLEAN_DIR / "products_clean.xlsx", sheet_name="products_clean", usecols=["product_id"])
CUSTOMER_IDS = set(customers["customer_id"])
PRODUCT_IDS = set(products["product_id"])

issue_log = pd.read_excel(ISSUE_LOG, sheet_name="issue_log")
m5_issues = issue_log[issue_log["milestone"] == "M5"]
m5_issues[["issue_id", "column", "issue", "rows_affected"]]

,issue_id,column,issue,rows_affected
55,ISS-056,customer_id,blank customer_id (anonymous visitor),150553
56,ISS-057,timestamp,timestamp not in standard ISO 8601 with timezone,60077
57,ISS-058,page_url,"URL with repeated slashes (e.g. 'http://///', '///')",31224
58,ISS-059,timestamp,blank value,20160
59,ISS-060,timestamp,impossible or unparseable timestamp,20011
60,ISS-061,timestamp,timestamp after the data export date (future),19919
61,ISS-062,device_id,leading/trailing spaces,9915
62,ISS-063,page_url,URL in upper case,8482
63,ISS-064,page_url,blank value,8305
64,ISS-065,page_url,URL without https:// scheme,8195


## 2. Cleaning rules

**URL** (`page_url`): all variants point to the same 5 page types of `shop.example.com`. Every URL is rebuilt to one canonical form:
1. lower case, then remove the scheme and any leading slashes (`http://///`, `HTTPS://`, none)
2. collapse repeated slashes, rewrite `/prod/` to `/product/`, remove trailing slashes
3. prefix `https://` (the site only serves https; every correct URL in the file uses it) and restore upper-case product IDs (`PROD-0109`)

The result must match one of the known page patterns, otherwise the run stops.

**Timestamp:** stored as UTC.
- Every timestamp that carries a timezone uses `+00:00`, so the pipeline writes UTC; the 20,102 values without a timezone are read as UTC too and marked in `timestamp_tz_assumed`.
- `2024/31/01 25:61:00` (month 31, hour 25, minute 61) is impossible: null, flagged.
- Events after the export day are impossible: null, flagged.

**IDs:** `device_id` is trimmed and normalised to a lower-case UUID with dashes (upper-case and dash-less forms are the same ID written differently); the placeholder `invalid-device-N` becomes null and is flagged. A blank `customer_id` is an anonymous visitor (normal for web traffic): kept as null and marked `is_anonymous`.

In [2]:
URL_PATTERN = re.compile(
    r"^https://shop\.example\.com/(|cart|search\?q=[a-z0-9+%]+|category/[a-z]+|product/PROD-\d{4})$"
)


def clean_url(s):
    u = s.str.strip().str.lower()
    u = u.str.replace(r"^(https?:)?/*", "", regex=True)       # scheme and leading slashes
    u = u.str.replace(r"/{2,}", "/", regex=True)              # repeated slashes
    u = u.str.replace("/prod/", "/product/", regex=False)     # wrong product path
    u = u.str.rstrip("/")                                     # trailing slashes
    u = "https://" + u.where(u.ne("shop.example.com"), "shop.example.com") + np.where(u.eq("shop.example.com"), "/", "")
    u = u.str.replace(r"prod-(\d+)", lambda m: f"PROD-{m.group(1)}", regex=True)
    return u.where(s.str.strip().ne(""))


def clean_chunk(chunk):
    """Clean one chunk; return (clean rows, flag rows, counters)."""
    out = pd.DataFrame({"event_id": chunk["event_id"], "session_id": chunk["session_id"]})
    n = {}

    cust = chunk["customer_id"].str.strip()
    out["customer_id"] = cust.where(cust.ne(""))
    out["is_anonymous"] = out["customer_id"].isna()
    out["event_type"] = chunk["event_type"].str.strip()

    url = clean_url(chunk["page_url"])
    bad = url.notna() & ~url.fillna("").str.match(URL_PATTERN)
    assert not bad.any(), chunk.loc[bad, "page_url"].unique()[:5]
    out["page_url"] = url
    out["product_id"] = url.str.extract(r"/product/(PROD-\d{4})$")[0]

    dev = chunk["device_id"]
    trimmed = dev.str.strip()
    n["device_trimmed"] = int(dev.ne(trimmed).sum())
    upper = trimmed.str.fullmatch(r"(?i)[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}") & ~trimmed.str.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")
    no_dash = trimmed.str.fullmatch(r"[0-9a-fA-F]{32}")
    placeholder_dev = trimmed.str.fullmatch(r"invalid-device-\d+")
    device = trimmed.str.lower().str.replace(
        r"^([0-9a-f]{8})([0-9a-f]{4})([0-9a-f]{4})([0-9a-f]{4})([0-9a-f]{12})$", r"\1-\2-\3-\4-\5", regex=True)
    out["device_id"] = device.where(~placeholder_dev)
    bad_dev = out["device_id"].notna() & ~out["device_id"].fillna("").str.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")
    assert not bad_dev.any(), chunk.loc[bad_dev, "device_id"].unique()[:5]
    n["device_upper"] = int(upper.sum())
    n["device_no_dash"] = int(no_dash.sum())
    n["device_placeholder"] = int(placeholder_dev.sum())

    raw_url = chunk["page_url"].str.strip()
    n["url_upper"] = int(raw_url.str.replace(r"PROD-\d+", "", regex=True).str.contains(r"[A-Z]").sum())
    n["url_no_scheme"] = int((raw_url.ne("") & ~raw_url.str.lower().str.startswith(("http://", "https://"))).sum())
    n["url_slashes"] = int(raw_url.str.replace(r"^https?://", "", regex=True).str.contains("//").sum())
    n["url_prod_path"] = int(raw_url.str.lower().str.contains("/prod/").sum())
    n["url_changed"] = int((raw_url.ne(out["page_url"].fillna(""))).sum())
    n["url_blank"] = int(raw_url.eq("").sum())

    ts = chunk["timestamp"].str.strip()
    has_tz = ts.str.contains(r"[+-]\d{2}:\d{2}$")
    parsed = pd.to_datetime(ts.str.replace("/", "-", regex=False), format="ISO8601", errors="coerce", utc=True)
    impossible = parsed.isna() & ts.ne("")
    future = parsed >= AFTER_SNAPSHOT
    out["timestamp_utc"] = parsed.where(~future).dt.tz_localize(None)
    out["timestamp_tz_assumed"] = parsed.notna() & ~future & ~has_tz
    n["ts_nonstandard"] = int((ts.ne("") & ~ts.str.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{6}\+00:00")).sum())
    n["ts_impossible"] = int(impossible.sum())
    n["ts_future"] = int(future.sum())
    n["ts_blank"] = int(ts.eq("").sum())
    n["ts_tz_assumed"] = int(out["timestamp_tz_assumed"].sum())

    out["ingest_run_id"] = chunk["ingest_run_id"].str.strip()

    flag_rows = pd.concat([
        pd.DataFrame({"event_id": chunk.loc[impossible, "event_id"], "column": "timestamp",
                      "raw_value": chunk.loc[impossible, "timestamp"], "reason": "impossible timestamp; set to null"}),
        pd.DataFrame({"event_id": chunk.loc[future, "event_id"], "column": "timestamp",
                      "raw_value": chunk.loc[future, "timestamp"], "reason": "timestamp after the data export (2025-12-02); set to null"}),
        pd.DataFrame({"event_id": chunk.loc[placeholder_dev, "event_id"], "column": "device_id",
                      "raw_value": chunk.loc[placeholder_dev, "device_id"], "reason": "placeholder device_id; set to null"}),
    ], ignore_index=True)
    n["orphan_customer"] = int((out["customer_id"].notna() & ~out["customer_id"].isin(CUSTOMER_IDS)).sum())
    n["orphan_product"] = int((out["product_id"].notna() & ~out["product_id"].isin(PRODUCT_IDS)).sum())
    return out, flag_rows, n


# quick look at what the URL rule does
demo = pd.Series(["HTTPS://SHOP.EXAMPLE.COM/PRODUCT/PROD-0001", "http://///shop.example.com/cart",
                  "shop.example.com/search?q=shoes///", "https://shop.example.com/prod////PROD-0143",
                  "https://shop.example.com////", ""])
pd.DataFrame({"raw": demo, "clean": clean_url(demo)})

,raw,clean
0,HTTPS://SHOP.EXAMPLE.COM/PRODUCT/PROD-0001,https://shop.example.com/product/PROD-0001
1,http://///shop.example.com/cart,https://shop.example.com/cart
2,shop.example.com/search?q=shoes///,https://shop.example.com/search?q=shoes
3,https://shop.example.com/prod////PROD-0143,https://shop.example.com/product/PROD-0143
4,https://shop.example.com////,https://shop.example.com/
5,,NaN


## 3. Process the full file chunk by chunk

In [3]:
clean_parts, flag_parts, counters = [], [], []
for i, chunk in enumerate(pd.read_csv(RAW_FILE, chunksize=CHUNK_SIZE, **READ_OPTS), start=1):
    clean, flagged, n = clean_chunk(chunk)
    clean_parts.append(clean)
    flag_parts.append(flagged)
    counters.append(n | {"chunk": i, "rows": len(chunk)})
    print(f"chunk {i}: {len(chunk):,} rows cleaned")

clickstream = pd.concat(clean_parts, ignore_index=True)
flag_table = pd.concat(flag_parts, ignore_index=True)
flag_table.insert(1, "flag_type", "invalid or missing value")  # impossible or future timestamps, set to null
totals = pd.DataFrame(counters).set_index("chunk")
print(f"\ntotal: {len(clickstream):,} rows")
totals

chunk 1: 100,000 rows cleaned


chunk 2: 100,000 rows cleaned


chunk 3: 100,000 rows cleaned


chunk 4: 100,000 rows cleaned


chunk 5: 100,000 rows cleaned

total: 500,000 rows


,device_trimmed,device_upper,device_no_dash,device_placeholder,url_upper,url_no_scheme,url_slashes,url_prod_path,url_changed,url_blank,ts_nonstandard,ts_impossible,ts_future,ts_blank,ts_tz_assumed,orphan_customer,orphan_product,rows
chunk,,,,,,,,,,,,,,,,,,
1,2009,1023,999,994,1678,1595,6222,1201,7817,1607,12091,3966,4040,4054,4019,0,0,100000
2,1988,1034,999,976,1695,1605,6194,1235,7799,1686,12074,4041,3835,3956,4067,0,0,100000
3,1987,1034,1017,1023,1664,1666,6283,1276,7949,1731,11963,4029,4063,3994,3977,0,0,100000
4,1921,941,982,1007,1729,1661,6248,1263,7909,1657,12073,4053,3916,4006,4043,0,0,100000
5,2010,1036,1033,986,1716,1668,6277,1238,7945,1624,11876,3922,4065,4150,3996,0,0,100000


## 4. Duplicates and relationships (checked across the whole file, not per chunk)

In [4]:
dup_event = clickstream["event_id"].duplicated()
dup_row = clickstream.duplicated()
print(f"duplicate event_id: {dup_event.sum():,} | exact duplicate rows: {dup_row.sum():,}")
print(f"events with unknown customer: {totals['orphan_customer'].sum():,} | unknown product: {totals['orphan_product'].sum():,}")
print(f"anonymous events: {clickstream['is_anonymous'].sum():,} ({clickstream['is_anonymous'].mean():.1%})")
assert not dup_event.any()

duplicate event_id: 0 | exact duplicate rows: 0
events with unknown customer: 0 | unknown product: 0
anonymous events: 150,553 (30.1%)


## 5. Fix log

In [5]:
t = totals.sum()
plan = [
    ("page_url", "URL in upper case", t["url_upper"], 0, "Lower-cased (product IDs kept upper case)"),
    ("page_url", "URL with repeated slashes", t["url_slashes"], 0, "Collapsed repeated slashes, removed trailing slashes"),
    ("page_url", "URL without https", t["url_no_scheme"], 0, "Added https://"),
    ("page_url", "non-standard product path", t["url_prod_path"], 0, "Rewrote /prod/ to /product/"),
    ("page_url", "blank value", 0, 0, "Kept as null (no source to fill)"),
    ("timestamp", "timestamp not in standard", t["ts_nonstandard"] - t["ts_impossible"], t["ts_tz_assumed"],
     "Parsed every format to UTC; values without timezone read as UTC and marked timestamp_tz_assumed"),
    ("timestamp", "impossible or unparseable", 0, t["ts_impossible"], "Set to null and flagged"),
    ("timestamp", "timestamp after the data export", 0, t["ts_future"], "Set to null and flagged"),
    ("timestamp", "blank value", 0, 0, "Kept as null (no source to fill)"),
    ("device_id", "leading/trailing spaces", t["device_trimmed"], 0, "Trimmed"),
    ("device_id", "device_id UUID in upper or mixed case", t["device_upper"], 0, "Lower-cased"),
    ("device_id", "device_id without dashes", t["device_no_dash"], 0, "Dashes inserted (8-4-4-4-12)"),
    ("device_id", "placeholder device_id", 0, t["device_placeholder"], "Set to null and flagged"),
    ("customer_id", "blank customer_id", 0, 0, "Kept as null; marked is_anonymous (not an error)"),
]
fix_rows = []
for column, start, fixed, flagged_n, action in plan:
    hit = m5_issues[(m5_issues["column"] == column) & m5_issues["issue"].str.startswith(start)]
    assert len(hit) == 1, (column, start)
    row = hit.iloc[0]
    fix_rows.append({"issue_id": row["issue_id"], "column": column, "issue": row["issue"],
                     "rows_in_m1": int(row["rows_affected"]), "rows_fixed": int(fixed),
                     "rows_flagged": int(flagged_n), "action_taken": action})
fix_log = pd.DataFrame(fix_rows)
assert set(fix_log["issue_id"]) == set(m5_issues["issue_id"]), "every M5 issue must be handled"
fix_log.drop(columns="issue")

,issue_id,column,rows_in_m1,rows_fixed,rows_flagged,action_taken
0,ISS-063,page_url,8482,8482,0,Lower-cased (product IDs kept upper case)
1,ISS-058,page_url,31224,31224,0,"Collapsed repeated slashes, removed trailing slashes"
2,ISS-065,page_url,8195,8195,0,Added https://
3,ISS-066,page_url,6213,6213,0,Rewrote /prod/ to /product/
4,ISS-064,page_url,8305,0,0,Kept as null (no source to fill)
5,ISS-057,timestamp,60077,40066,20102,Parsed every format to UTC; values without timezone read as UTC an...
6,ISS-060,timestamp,20011,0,20011,Set to null and flagged
7,ISS-061,timestamp,19919,0,19919,Set to null and flagged
8,ISS-059,timestamp,20160,0,0,Kept as null (no source to fill)
9,ISS-062,device_id,9915,9915,0,Trimmed


## 6. Validation

In [6]:
ts = clickstream["timestamp_utc"].dropna()
checks = {
    "500,000 events, event_id unique": len(clickstream) == 500_000 and clickstream["event_id"].is_unique,
    "every URL canonical or null": clickstream["page_url"].dropna().str.match(URL_PATTERN).all(),
    "URL nulls = raw blanks only": clickstream["page_url"].isna().sum() == t["url_blank"],
    "timestamps between first event and export day":
        ts.between(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-02 23:59:59")).all(),
    "device_id is a lower-case UUID or null": clickstream["device_id"].dropna().str.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}").all(),
    "event_type in 4 known values": clickstream["event_type"].isin(["page_view", "search", "add_to_cart", "login"]).all(),
    "every known customer_id exists in customers": t["orphan_customer"] == 0,
    "every product in a URL exists in products": t["orphan_product"] == 0,
    "fits in one Excel sheet": len(clickstream) < 1_048_576,
}
validation = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
assert validation["passed"].all(), validation
print(f"timestamp range: {ts.min()} to {ts.max()} (UTC)")
validation

timestamp range: 2025-09-02 16:05:22.021519 to 2025-12-02 16:04:29.908528 (UTC)


,check,passed
0,"500,000 events, event_id unique",True
1,every URL canonical or null,True
2,URL nulls = raw blanks only,True
3,timestamps between first event and export day,True
4,device_id is a lower-case UUID or null,True
5,event_type in 4 known values,True
6,every known customer_id exists in customers,True
7,every product in a URL exists in products,True
8,fits in one Excel sheet,True


In [7]:
completeness = clickstream.notna().mean().mul(100).round(2).rename("% filled")
print(completeness.to_string())
print()
print(clickstream["page_url"].str.replace(r"PROD-\d{4}", "PROD-####", regex=True).value_counts(dropna=False).to_string())
clickstream.head(8)

event_id                100.00
session_id              100.00
customer_id              69.89
is_anonymous            100.00
event_type              100.00
page_url                 98.34
product_id               73.79
device_id                99.00
timestamp_utc            87.98
timestamp_tz_assumed    100.00
ingest_run_id           100.00



page_url
https://shop.example.com/product/PROD-####       368942
https://shop.example.com/cart                     30880
https://shop.example.com/category/electronics     30762
https://shop.example.com/                         30653
https://shop.example.com/search?q=shoes           30458
NaN                                                8305


,event_id,session_id,customer_id,is_anonymous,event_type,page_url,product_id,device_id,timestamp_utc,timestamp_tz_assumed,ingest_run_id
0,aa56e7fb-5d80-4807-86dc-1f742f655fbe,8ee809b4-5d5b-4fed-99c6-67a74e8fe3e9,NaN,True,login,https://shop.example.com/search?q=shoes,NaN,9a921b19-37df-4ed4-85d0-8583864f8d1d,2025-09-26 16:03:31.922468,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
1,9ea31e17-ffce-4d2b-93f2-979fafac5b9b,9efcf371-ae3b-4a9e-956b-67dbc09935c0,ea0320ec-675d-42be-bcf8-ec2b2a71b14b,False,page_view,https://shop.example.com/product/PROD-0109,PROD-0109,06e4844b-068b-41f4-9c78-120424a84b72,2025-10-15 12:27:28.922583,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
2,dadd5a4b-3654-440d-adb1-b7a3ff181da3,51d6406d-abd3-4f96-8fe9-039088694420,5160a713-dde1-404a-8b29-f3cc1192a694,False,login,https://shop.example.com/,NaN,8bcca371-31ce-4420-801f-6b18a7512463,2025-09-03 07:46:34.922628,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
3,c0f6a7b0-f688-4aac-8740-11821f7f086e,41a4433e-fcf4-4f20-8264-ddbdd0bfb567,d8fde909-fe69-4f34-9290-cebfc268c31c,False,search,https://shop.example.com/,NaN,f0241556-af81-429b-ab06-e19a4ec37f2b,NaT,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
4,e0cc5a38-1047-4d47-9679-7238b3ff9e79,ec4f96c4-82c5-4622-bbba-961efa314a9a,fc905a7d-b0b4-4217-ad84-c300dbfdeac9,False,page_view,https://shop.example.com/product/PROD-0143,PROD-0143,6dd78212-572a-41c4-88e8-deded58c3bb6,2025-10-06 19:03:15.922725,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
5,ce05f214-1432-4d06-954e-816086edab0a,6f92abb3-8e6b-4d75-a8e8-0c52ec905135,5d4e1b1c-e9d9-4c0c-8c39-142dc1f5f5c7,False,page_view,https://shop.example.com/product/PROD-0038,PROD-0038,f0c53887-5ec6-4ef9-a91b-2f8d08eb763d,2025-09-21 00:21:39.922755,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
6,12c101c6-33ed-4002-b473-b3286e606a3f,cef0a6ec-428e-4561-8786-3aa6ad7c0d2b,661810c5-91ac-4d99-8791-d25c7a977a8e,False,page_view,https://shop.example.com/product/PROD-0279,PROD-0279,8550dce6-f181-4b4f-bd49-2f1a9db57491,2025-09-25 18:14:30.922780,False,920fe448-95d0-4ccb-b77b-2c7cfa389054
7,d3b8aa2b-4fbd-4232-b35f-0a30cbfa120f,21930c3a-0bda-4c08-886c-f7086fcff8b9,9e0f90fc-2741-4828-907b-64b434a5e9e2,False,page_view,https://shop.example.com/product/PROD-0425,PROD-0425,67da5d36-1a20-4181-9d8b-c9587bf96439,NaT,False,920fe448-95d0-4ccb-b77b-2c7cfa389054


## 7. Save outputs

After saving, the clean workbook is **read back** and compared with the data in memory (row count and filled cells per column), so a silent write problem cannot go unnoticed.

In [8]:
def write_big_xlsx(path, sheets):
    # strings_to_urls off: Excel allows only 65,530 hyperlinks per sheet, so URLs are stored as plain text
    with pd.ExcelWriter(path, engine="xlsxwriter", datetime_format="yyyy-mm-dd hh:mm:ss",
                        engine_kwargs={"options": {"strings_to_urls": False}}) as writer:
        for sheet, data in sheets.items():
            data.to_excel(writer, sheet_name=sheet, index=False)
            ws = writer.sheets[sheet]
            ws.freeze_panes(1, 0)
            for i, col in enumerate(data.columns):
                ws.set_column(i, i, min(max(data[col].head(2000).astype(str).str.len().max(), len(col)) + 2, 60))
    print(f"saved: {path.resolve()} ({path.stat().st_size / 1024**2:.1f} MB)")


write_big_xlsx(CLEAN_DIR / "clickstream_clean.xlsx", {
    "clickstream": clickstream,
    "flagged": flag_table.sort_values(["flag_type", "reason", "event_id"]),
})

# CSV: 500,000 rows are larger than GitHub's 100 MB file limit as plain text, so the CSV is zipped
CSV_DIR = CLEAN_DIR / "csv"
CSV_DIR.mkdir(exist_ok=True)
clickstream.to_csv(CSV_DIR / "clickstream_clean.zip", index=False, encoding="utf-8",
                   date_format="%Y-%m-%d %H:%M:%S.%f",
                   compression={"method": "zip", "archive_name": "clickstream_clean.csv"})
print(f"saved CSV: clickstream_clean.zip ({(CSV_DIR / 'clickstream_clean.zip').stat().st_size / 1024**2:.1f} MB)")

with pd.ExcelWriter(OUT_DIR / "m5_fix_log.xlsx", engine="xlsxwriter") as writer:
    for name, data in {"fix_log": fix_log, "validation": validation, "chunks": totals.reset_index()}.items():
        data.to_excel(writer, sheet_name=name, index=False)
        writer.sheets[name].freeze_panes(1, 0)
        for i, col in enumerate(data.columns):
            writer.sheets[name].set_column(i, i, min(max(data[col].astype(str).str.len().max(), len(col)) + 2, 70))
print("saved:", (OUT_DIR / "m5_fix_log.xlsx").resolve())

saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\03_clean_data_and_report\clean_data\clickstream_clean.xlsx (52.5 MB)


saved CSV: clickstream_clean.zip (46.7 MB)
saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\02_process\05_cleaning_clickstream-pandas\output\m5_fix_log.xlsx


In [9]:
check = pd.read_excel(CLEAN_DIR / "clickstream_clean.xlsx", sheet_name="clickstream", dtype=str)
expected = clickstream.notna().sum()
read_back = check.notna().sum()
assert len(check) == len(clickstream), "row count differs"
assert read_back.equals(expected), pd.DataFrame({"in memory": expected, "in file": read_back})
assert check["event_id"].tolist() == clickstream["event_id"].tolist()
print(f"read-back OK: {len(check):,} rows, filled cells per column identical to memory")

read-back OK: 500,000 rows, filled cells per column identical to memory
